In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/FoodLens/food-classifier-code.zip" 

!unzip -q "$DRIVE_ZIP_PATH" -d /content/
%cd /content/food-classifier-code
!ls

In [ ]:
!pip install -q huggingface_hub

import os, re, shutil, random
from huggingface_hub import snapshot_download

RAW_DIR = snapshot_download(
    repo_id="SKSKCulinaryAI/Indian_Food_Images_Dataset",
    repo_type="dataset",
    local_dir="/content/raw_dataset",
)
SRC = os.path.join(RAW_DIR, "train")
print("Downloaded to:", SRC)
print("Classes found:", os.listdir(SRC))

In [ ]:
def normalize_name(name):
    """'Butter chicken' -> 'butter_chicken' — matches model/labels.txt"""
    name = name.strip().lower().replace(" ", "_")
    name = re.sub(r"[^a-z0-9_]", "", name)
    return name

DEST = "dataset"
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}
random.seed(42)

for split in SPLIT:
    os.makedirs(f"{DEST}/{split}", exist_ok=True)

classes = sorted(os.listdir(SRC))
for cls in classes:
    cls_dir = os.path.join(SRC, cls)
    if not os.path.isdir(cls_dir):
        continue
    norm_cls = normalize_name(cls)
    files = sorted([f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    random.shuffle(files)
    n = len(files)
    n_train = int(n * SPLIT["train"])
    n_val = int(n * SPLIT["val"])
    buckets = {
        "train": files[:n_train],
        "val": files[n_train:n_train + n_val],
        "test": files[n_train + n_val:],
    }
    for split, split_files in buckets.items():
        out_dir = f"{DEST}/{split}/{norm_cls}"
        os.makedirs(out_dir, exist_ok=True)
        for fname in split_files:
            shutil.copy(os.path.join(cls_dir, fname), os.path.join(out_dir, fname))
    print(f"{cls} -> {norm_cls}: {n} images ({len(buckets['train'])} train / {len(buckets['val'])} val / {len(buckets['test'])} test)")

print("\nSplit complete. Classes:", sorted(os.listdir(f'{DEST}/train')))

In [ ]:
!python train.py --dataset dataset --backbone efficientnetb0 \
    --head-epochs 8 --finetune-epochs 20 --batch-size 32

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/FoodLens/trained_model"  # <-- apna path yaha daalo
!mkdir -p "$SAVE_DIR"
!cp model/food_classifier.h5 "$SAVE_DIR/"
!cp model/labels.txt "$SAVE_DIR/"
print("Saved to Drive:", SAVE_DIR)

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image

model = load_model("model/food_classifier.h5")
with open("model/labels.txt") as f:
    labels = [l.strip() for l in f if l.strip()]

TEST_IMAGE = "dataset/test/biryani/REPLACE_WITH_REAL_FILENAME.jpg"  # <-- change this

img = Image.open(TEST_IMAGE).convert("RGB").resize((224, 224))
arr = preprocess_input(np.asarray(img, dtype="float32"))
arr = np.expand_dims(arr, axis=0)

probs = model.predict(arr)[0]
top_idx = np.argsort(probs)[::-1][:3]
for i in top_idx:
    print(f"{labels[i]}: {probs[i]*100:.2f}%")